# Figure S9

Stable-denominator common-gray results are separate from the 21-condition baseline and recurrent perturbation source families.


## 1. Imports and configuration
Artifact-only replay; no models or acquisition Notebook execution.


In [1]:
# Colab needs the repository's saved analysis inputs, not just the package.
import importlib
import importlib.util
import os
import site
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path("/content/flyvis-midd")
    if not (ROOT / "src/flyvis_midd").is_dir():
        if ROOT.exists():
            raise RuntimeError("/content/flyvis-midd exists but is not a repository checkout.")
        result = subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main",
             "https://github.com/poxsalop5029/flyvis-midd.git", str(ROOT)],
            capture_output=True, text=True,
        )
        if result.returncode:
            message = result.stderr or result.stdout
            raise RuntimeError("Could not clone the public flyvis-midd repository: " + message[-2000:])
    missing = [name for name in ("numpy", "pandas", "matplotlib")
               if importlib.util.find_spec(name) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
        for package_dir in site.getsitepackages():
            site.addsitedir(package_dir)
    importlib.invalidate_caches()
else:
    ROOT = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / "src/flyvis_midd").is_dir()), None,
    )
    if ROOT is None:
        raise RuntimeError("Run this notebook from inside the flyvis-midd repository.")
sys.path.insert(0, str(ROOT / "src"))
print("Repository inputs:", ROOT)


from IPython.display import display
from flyvis_midd.figure_analysis import (
    input_manifest, load_tables, validate_inputs, primary_analysis,
    adopted_analysis, statistical_tables, write_reproduction_record,
)
from flyvis_midd.figure_rendering import render
FIGURE = 'S9'

from flyvis_midd.figure_analysis import table


Repository inputs: <repository-root>


## 2. Input artifact manifest
Source paths and SHA-256 identify the immutable numerical inputs. Bundled precomputed artifacts are numerical inputs.


In [2]:
manifest = input_manifest(FIGURE, ROOT)
display(manifest)


,path,sha256,bytes
0,data/precomputed/fig06/...,e3ec244647fb0594d99b3c77763414b9ebfd3cdedfe3a7...,13238
1,data/precomputed/fig06/exp09_coupling_by_model...,f6f1580d8a91cbff04baef62344436e1574a038d6de7b1...,64412
2,data/precomputed/fig06/exp10_model_level_stati...,910f4744e05d4f4e6d8d5ed74478f5a9790f51d6674c75...,14886
3,data/precomputed/fig06/mve02_sintel_specificit...,c875c3fca4ee63e9713c362224ae598cd18af77fd41042...,1545
4,data/precomputed/fig06/exp09_activity_by_model...,bf42d6be9eb73bc07ece9f6ae8f5e5590a319549bf0666...,529438
5,data/precomputed/fig06/mve02_midd_matched_rand...,913fd9f63d771aab277d9b147c683a5683fa8a1fce7ee7...,10382124
6,data/precomputed/fig06/...,12f2b7195d111615289cad69ad4311af12cf563f4608df...,5329
7,data/precomputed/fig06/...,e1ea0c57a146da3423ca47a0a7715ee107d40116e883a4...,3249
8,data/precomputed/fig03/...,af9bcc64596db095eb3198ee30c11b2e4a120e988f4370...,29693
9,data/precomputed/fig03/...,ac03e9ec916b620975ac2b2132b72eb302221ab9b3d477...,9804


## 3. Canonical data load
Each source keeps its cohort, renderer, operator, temporal support, and metric.


In [3]:
tables = load_tables(manifest, ROOT)
print("Loaded source tables:", len(tables))


Loaded source tables: 12


## 4. QA and assertions
Check within-contract A/S algebra and adopted source equivalence before saving.


In [4]:
integration_checks = validate_inputs(tables)
display(integration_checks)


,check,passed
0,A/S identity: data/precomputed/fig06/mve02_mid...,True


## 5. Primary analysis
For Fig3/S7, affine fits and both held-out splits are recomputed from saved A3/A4. Native and multi-field definitions remain separate.


In [5]:
primary = primary_analysis(FIGURE, tables)
for name, frame in primary.items():
    print(name)
    display(frame.head(20))


paired_noop_normalized_changes


,model_id,intervention,condition,baseline,intervention_value,G_noop,raw_delta,normalized_change,baseline_aligned_attenuation,projected_MIDD_gain,RMS_normalized_difference
0,0,T4,MIDD,-0.001862,-0.013206,0.189787,-0.011344,-0.059772,-0.059772,-1.112365,3.048947
1,0,T4,propeller,9.756061,6.612944,10.712049,-3.143117,-0.293419,0.293419,-1.112365,3.048947
2,0,T4,edge,0.832073,0.813430,1.153256,-0.018644,-0.016166,0.016166,-1.112365,3.048947
3,0,T4_T5,MIDD,-0.001862,-0.001399,0.189787,0.000463,0.002439,0.002439,0.183834,0.852881
4,0,T4_T5,propeller,9.756061,4.006075,10.712049,-5.749985,-0.536777,0.536777,0.183834,0.852881
5,0,T4_T5,edge,0.832073,0.761638,1.153256,-0.070436,-0.061075,0.061075,0.183834,0.852881
6,0,T4a,MIDD,-0.001862,-0.001900,0.189787,-0.000038,-0.000200,-0.000200,0.397955,0.703503
7,0,T4a,propeller,9.756061,8.184284,10.712049,-1.571777,-0.146730,0.146730,0.397955,0.703503
8,0,T4a,edge,0.832073,0.395950,1.153256,-0.436123,-0.378167,0.378167,0.397955,0.703503
9,0,T4b,MIDD,-0.001862,-0.005152,0.189787,-0.003290,-0.017334,-0.017334,0.975361,0.490710


## 6. Adopted validation and controls
Use frozen source records for matched controls, geometry, learning, interventions, and duration. Do not substitute historical intervention estimates for the common-gray operation.


In [6]:
adopted = adopted_analysis(FIGURE, tables)
for name, frame in adopted.items():
    print(name)
    display(frame.head(20))


data/precomputed/fig06/replacement_paired_losses.csv


,model_id,intervention,A_C48_noop,A_C48_intervention,MIDD_loss,propeller_loss,edge_loss,propeller_magnitude_loss,MIDD_absolute_loss,projected_MIDD_loss,D_primary,D_edge,D_projected,denominators_valid,A_sign_flip
0,0,T4,-0.001862,-0.013206,-6.092053,0.322171,0.022406,0.198639,-6.092053,2.112365,-6.414224,-6.114460,1.790194,True,False
1,0,T4_T5,-0.001862,-0.001399,0.248546,0.589376,0.084651,0.489372,0.248546,0.816166,-0.340830,0.163895,0.226791,True,False
2,0,T4a,-0.001862,-0.001900,-0.020419,0.161108,0.524140,0.055375,-0.020419,0.602045,-0.181527,-0.544560,0.440937,True,False
3,0,T4b,-0.001862,-0.005152,-1.766716,0.189212,-0.029355,-0.030239,-1.766716,0.024639,-1.955928,-1.737361,-0.164573,True,False
4,0,all_input,-0.001862,0.000000,1.000000,1.000000,1.000000,0.986147,1.000000,1.000000,0.000000,0.000000,0.000000,True,False
5,0,sham,-0.001862,-0.001553,0.166192,0.087072,0.167484,0.079138,0.166192,0.013509,0.079120,-0.001293,-0.073562,True,False
6,1,T4,-0.001428,-0.000064,0.955027,0.759028,0.788907,0.298910,0.955027,0.756204,0.196000,0.166121,-0.002824,True,False
7,1,T4_T5,-0.001428,0.000092,1.064399,0.919804,0.853966,0.606969,0.935601,0.871811,0.144596,0.210434,-0.047993,True,True
8,1,T4a,-0.001428,-0.001835,-0.284621,0.190935,1.045217,0.021094,-0.284621,0.155034,-0.475556,-1.329838,-0.035901,True,False
9,1,T4b,-0.001428,-0.001422,0.004366,0.215717,0.021275,-0.017355,0.004366,0.316807,-0.211351,-0.016909,0.101089,True,False


data/precomputed/fig06/replacement_qa.csv


,model_id,group,max_error,passed,check
0,0,3MIDD_048,6.355295e-08,True,pre_extension_canonical
1,0,3MIDD_207,2.538126e-08,True,pre_extension_canonical
2,0,4MIDD_085_170,2.009836e-08,True,pre_extension_canonical
3,0,NaN,NaN,True,noop_Static
4,0,NaN,NaN,True,noop_CW
5,0,NaN,NaN,True,noop_CCW
6,0,NaN,NaN,True,noop_Right
7,0,NaN,NaN,True,noop_Left
8,0,NaN,NaN,True,noop_MIDD_0_original
9,0,NaN,NaN,True,noop_MIDD_0_mirror


data/precomputed/fig06/replacement_bootstrap.csv


...

,intervention,condition,metric,mean,median,ci_low,ci_high
0,T4,MIDD,normalized_change,0.054487,0.049030,0.013611,0.100006
1,T4,MIDD,baseline_aligned_attenuation,0.051874,0.049030,0.010934,0.098355
2,T4,MIDD,raw_delta,0.007052,0.007322,0.001366,0.012364
3,T4,MIDD,projected_MIDD_gain,0.147913,0.219336,-0.177908,0.405746
4,T4,MIDD,RMS_normalized_difference,1.073006,0.842099,0.800362,1.536004
5,T4,edge,normalized_change,-0.344956,-0.450122,-0.572562,-0.065389
6,T4,edge,baseline_aligned_attenuation,0.344956,0.450122,0.064527,0.574440
7,T4,edge,raw_delta,-0.438624,-0.488432,-0.702564,-0.157349
8,T4,edge,projected_MIDD_gain,0.147913,0.219336,-0.183183,0.400032
9,T4,edge,RMS_normalized_difference,1.073006,0.842099,0.801035,1.532324


data/precomputed/fig06/denominator_audit.csv


,intervention,condition,q0,q5,q25,q50,q75,q95,q100,n,excluded
0,T4,MIDD,0.115196,0.116270,0.121877,0.157965,0.200315,0.235772,0.253071,10,0
1,T4,edge,0.759230,0.813264,0.953862,1.134334,1.260944,1.654653,1.857701,10,0
2,T4,propeller,9.085153,9.288611,10.627711,11.237086,11.801027,13.651906,13.796480,10,0
3,T4_T5,MIDD,0.115196,0.116270,0.121877,0.157965,0.200315,0.235772,0.253071,10,0
4,T4_T5,edge,0.759230,0.813264,0.953862,1.134334,1.260944,1.654653,1.857701,10,0
5,T4_T5,propeller,9.085153,9.288611,10.627711,11.237086,11.801027,13.651906,13.796480,10,0
6,T4a,MIDD,0.115196,0.116270,0.121877,0.157965,0.200315,0.235772,0.253071,10,0
7,T4a,edge,0.759230,0.813264,0.953862,1.134334,1.260944,1.654653,1.857701,10,0
8,T4a,propeller,9.085153,9.288611,10.627711,11.237086,11.801027,13.651906,13.796480,10,0
9,T4b,MIDD,0.115196,0.116270,0.121877,0.157965,0.200315,0.235772,0.253071,10,0


data/precomputed/fig06/replacement_metrics.csv


,model_id,condition,intervention,mean_flow_magnitude,mean_signed_tangent,mean_abs_tangent,signed_tangent_ratio,rotation_coherence,mean_signed_radial,annulus_positions,annulus_receptors,mean_u
0,0,Static,noop,0.275658,-0.107649,0.139993,-0.390517,0.768963,0.178290,698,698,-0.076315
1,0,Static,T4a,0.387465,-0.099979,0.189618,-0.258034,0.527265,0.174250,698,698,-0.008540
2,0,Static,T4b,0.500321,-0.024984,0.304798,-0.049936,0.081969,0.160870,698,698,-0.400876
3,0,Static,T4,1.616435,0.149755,0.996580,0.092645,0.150269,0.386532,698,698,-0.743651
4,0,Static,T4_T5,1.354645,-0.021478,0.803792,-0.015855,0.026721,0.611023,698,698,-0.493896
5,0,Static,sham,0.269857,-0.085025,0.137967,-0.315075,0.616273,0.144984,698,698,-0.153208
6,0,Static,all_input,0.148397,0.032469,0.075630,0.218801,0.429321,0.084941,698,698,-0.092684
7,0,CW,noop,10.784526,9.815791,9.960090,0.910174,0.985512,1.365292,698,698,0.445993
8,0,CW,T4a,10.126642,8.102660,8.523523,0.800133,0.950623,1.450428,698,698,3.169957
9,0,CW,T4b,11.227115,7.922450,9.241721,0.705653,0.857248,1.789472,698,698,-3.540715


### A. Common-baseline controls
The matched-control analysis uses the same gray decoder-input replacement for MIDD, propeller, and moving edge. Model-wise paired losses remain the primary comparison.

In [7]:
for suffix in ['paired_losses.csv']:
    print(suffix)
    display(table(tables, suffix).head(20))

paired_losses.csv


,model_id,intervention,A_C48_noop,A_C48_intervention,MIDD_loss,propeller_loss,edge_loss,propeller_magnitude_loss,MIDD_absolute_loss,projected_MIDD_loss,D_primary,D_edge,D_projected,denominators_valid,A_sign_flip
0,0,T4,-0.001862,-0.013206,-6.092053,0.322171,0.022406,0.198639,-6.092053,2.112365,-6.414224,-6.114460,1.790194,True,False
1,0,T4_T5,-0.001862,-0.001399,0.248546,0.589376,0.084651,0.489372,0.248546,0.816166,-0.340830,0.163895,0.226791,True,False
2,0,T4a,-0.001862,-0.001900,-0.020419,0.161108,0.524140,0.055375,-0.020419,0.602045,-0.181527,-0.544560,0.440937,True,False
3,0,T4b,-0.001862,-0.005152,-1.766716,0.189212,-0.029355,-0.030239,-1.766716,0.024639,-1.955928,-1.737361,-0.164573,True,False
4,0,all_input,-0.001862,0.000000,1.000000,1.000000,1.000000,0.986147,1.000000,1.000000,0.000000,0.000000,0.000000,True,False
5,0,sham,-0.001862,-0.001553,0.166192,0.087072,0.167484,0.079138,0.166192,0.013509,0.079120,-0.001293,-0.073562,True,False
6,1,T4,-0.001428,-0.000064,0.955027,0.759028,0.788907,0.298910,0.955027,0.756204,0.196000,0.166121,-0.002824,True,False
7,1,T4_T5,-0.001428,0.000092,1.064399,0.919804,0.853966,0.606969,0.935601,0.871811,0.144596,0.210434,-0.047993,True,True
8,1,T4a,-0.001428,-0.001835,-0.284621,0.190935,1.045217,0.021094,-0.284621,0.155034,-0.475556,-1.329838,-0.035901,True,False
9,1,T4b,-0.001428,-0.001422,0.004366,0.215717,0.021275,-0.017355,0.004366,0.316807,-0.211351,-0.016909,0.101089,True,False


### B. Historical activity association
Activity associations are observational and are separate from the matched-control intervention contract.

In [8]:
for suffix in ['exp09_coupling_by_model.csv', 'exp09_activity_by_model_luminance.csv']:
    print(suffix)
    display(table(tables, suffix).head(20))

exp09_coupling_by_model.csv


,model_id,cell_type,rho_signed,rho_absolute,sign_agreement,zero_nearest_cell,zero_nearest_flow,response_identifiable,mean_spatial_cosine,mean_mirror_antisymmetry,intercept,slope,r2,rmse
0,0,T2,0.785714,0.785714,0.428571,207,143,False,0.008348,0.005282,-0.062251,22.834915,0.766712,0.023847
1,0,T2a,0.857143,0.857143,0.857143,175,143,False,0.000289,0.000000,-0.008741,471.754256,0.814226,0.021280
2,0,T3,-0.250000,0.250000,0.571429,207,143,True,-0.011495,0.764770,-0.073103,-55.310784,0.104602,0.046718
3,0,T4a,0.892857,0.892857,0.428571,207,143,True,0.008624,0.839443,-0.144155,64.849776,0.984945,0.006058
4,0,T4b,0.964286,0.964286,0.857143,143,143,False,0.002643,0.438709,-0.010330,30.666586,0.988440,0.005308
5,0,T4c,-0.964286,0.964286,0.000000,112,143,True,0.001340,0.616878,0.015980,-19.653853,0.978976,0.007159
6,0,T4d,-0.500000,0.500000,0.142857,80,143,True,0.001852,0.100442,0.038032,-81.765858,0.369083,0.039216
7,0,T5a,0.892857,0.892857,0.857143,175,143,False,0.006010,0.057686,-0.023540,30.474208,0.900375,0.015583
8,0,T5b,0.892857,0.892857,1.000000,207,143,False,0.001883,0.003325,-0.006605,50.245194,0.908673,0.014920
9,0,T5c,-0.892857,0.892857,0.571429,207,143,True,-0.018344,0.815446,-0.084743,-28.133462,0.994235,0.003749


exp09_activity_by_model_luminance.csv


,model_id,cell_type,c,e_raw,e_original_projected,e_mirror_projected,e_negative_control,spatial_cosine,mirror_antisymmetry,n_global_positions,pd_resultant_lomo,response_scale_rms,control_response_threshold,e_normalized,response_identifiable
0,0,T2,48,0.003588,-0.005247,-0.012423,-0.008992,0.009928,0.000000,2790,0.887383,0.003620,0.008992,0.991050,False
1,0,T2,80,0.005556,-0.001726,-0.012838,-0.007416,0.015170,0.000000,2790,0.887383,0.003620,0.008992,1.534648,False
2,0,T2,112,0.005586,0.000207,-0.010966,-0.005422,0.015070,0.036975,2790,0.887383,0.003620,0.008992,1.542938,False
3,0,T2,143,0.003459,-0.000352,-0.007271,-0.003407,0.009241,0.000000,2790,0.887383,0.003620,0.008992,0.955423,False
4,0,T2,175,0.001772,-0.000586,-0.004131,-0.001517,0.004697,0.000000,2790,0.887383,0.003620,0.008992,0.489562,False
5,0,T2,191,0.001240,-0.000483,-0.002962,-0.000903,0.003275,0.000000,2790,0.887383,0.003620,0.008992,0.342399,False
6,0,T2,207,0.000401,-0.000744,-0.001545,-0.000284,0.001055,0.000000,2790,0.887383,0.003620,0.008992,0.110626,False
7,0,T2a,48,0.000141,0.000376,0.000095,0.000353,0.001127,0.000000,2790,0.369438,0.000101,0.000379,1.393276,False
8,0,T2a,80,0.000205,0.000461,0.000051,0.000312,0.001627,0.000000,2790,0.369438,0.000101,0.000379,2.024293,False
9,0,T2a,112,0.000017,0.000223,0.000188,0.000359,0.000138,0.000000,2790,0.369438,0.000101,0.000379,0.172571,False


### C. Historical replacement
The historical 21-condition replacement has a different baseline and is not pooled with common-gray paired losses.

In [9]:
for suffix in ['exp10_model_level_statistics_corrected.csv']:
    print(suffix)
    display(table(tables, suffix).head(20))

exp10_model_level_statistics_corrected.csv


,intervention_id,mean_loss,ci95_low,ci95_high,median_loss,sd_loss,n_loss_positive,p_signflip,mean_shape_residual,mean_propeller_loss,mean_specificity,kind,method,cell_types,family,inference_family,q_prespecified_family
0,control__all34__baseline,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,10,0.001951,0.000000e+00,1.000000,0.000000e+00,control,baseline,T1|T2|T2a|T3|T4a|T4b|T4c|T4d|T5a|T5b|T5c|T5d|T...,controls,controls,0.001951
1,control__all34__zero,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,10,0.001951,0.000000e+00,1.000000,0.000000e+00,control,zero,T1|T2|T2a|T3|T4a|T4b|T4c|T4d|T5a|T5b|T5c|T5d|T...,controls,controls,0.001951
2,control__noop,1.365574e-15,9.214851e-16,1.876277e-15,1.276756e-15,8.242778e-16,10,0.001951,8.918803e-16,0.000000,1.365574e-15,control,noop,NaN,controls,controls,0.001951
3,control__sham,1.365574e-15,9.214851e-16,1.876277e-15,1.276756e-15,8.242778e-16,10,0.001951,8.918803e-16,0.000000,1.365574e-15,control,sham,NaN,controls,controls,0.001951
4,group__T4_T5__baseline,6.281187e-01,2.636447e-01,9.094720e-01,8.146622e-01,5.613993e-01,9,0.006829,3.482151e-01,0.608594,1.952503e-02,group,baseline,T4a|T4b|T4c|T4d|T5a|T5b|T5c|T5d,groups,prespecified_groups,0.017073
5,group__T4_family__baseline,3.867609e-01,8.699287e-02,6.602980e-01,4.854555e-01,4.897838e-01,8,0.018537,4.633981e-01,0.523298,-1.365375e-01,group,baseline,T4a|T4b|T4c|T4d,groups,prespecified_groups,0.030894
6,group__T5_family__baseline,3.903721e-02,-7.895861e-02,1.805629e-01,1.367888e-02,2.212674e-01,5,0.323902,2.460667e-01,0.117896,-7.885919e-02,group,baseline,T5a|T5b|T5c|T5d,groups,prespecified_groups,0.323902
7,group__formal_candidates__baseline,1.310806e-01,-1.185251e-01,3.689800e-01,2.067920e-01,4.142433e-01,7,0.170732,3.661687e-01,0.296926,-1.658458e-01,group,baseline,T4a|TmY14|T2a|Tm16,groups,prespecified_groups,0.213415
8,group__formal_candidates_plus_T4b__baseline,4.943027e-01,2.894050e-01,7.101164e-01,4.870294e-01,3.597412e-01,10,0.001951,4.499603e-01,0.375332,1.189704e-01,group,baseline,T4a|TmY14|T2a|Tm16|T4b,groups,prespecified_groups,0.009756
9,single__T1__baseline,-2.943943e-01,-7.379929e-01,2.432923e-02,-2.709533e-02,6.672882e-01,3,0.887805,3.719104e-01,0.067833,-3.622274e-01,single,baseline,T1,screen,exploratory_34_screen,0.986341


### D. Recurrent subtype intervention
Recurrent interventions and their Sintel comparison are a separate manipulation, not the decoder-input replacement.

In [10]:
for suffix in ['mve02_sintel_specificity.csv', 'mve02_midd_matched_random.csv']:
    print(suffix)
    display(table(tables, suffix).head(20))

mve02_sintel_specificity.csv


,seed,cohort,set_name,delta_EPE,random_mean_delta_EPE,Z_flow,p_flow,Z_MIDD,relative_Z
0,0,comparator,T4c,0.563759,0.381060,5.462674,0.005,-11.987567,-17.450241
1,0,comparator,T4d,0.542505,0.382145,4.924041,0.005,-5.926494,-10.850535
2,1,established,T4c,0.542732,0.078055,62.987739,0.005,-13.645825,-76.633564
3,1,established,T4d,0.112637,0.078289,4.422555,0.005,-0.154693,-4.577248
4,2,established,T4c,1.087386,0.281102,25.814712,0.005,0.296080,-25.518632
5,2,established,T4d,3.546385,0.277982,118.479367,0.005,8.341981,-110.137386
6,3,established,T4c,0.056825,0.061723,-1.055224,0.855,-6.036686,-4.981462
7,3,established,T4d,0.092492,0.061263,6.312921,0.005,-1.811946,-8.124867
8,8,comparator,T4c,0.622449,0.204442,16.097437,0.005,-8.904868,-25.002305
9,8,comparator,T4d,0.287763,0.205160,3.094434,0.010,-2.047371,-5.141805


mve02_midd_matched_random.csv


,seed,cohort,holdout,optimizer_step,phase_deg,set_name,control,draw,n_nodes,baseline_A,...,delta_M,E_reference,E_abs_A,T_original,T_mirror,A,abs_A,S,M,actual_reversal
0,1,established,False,250000,0.0,T4c,baseline,-2,0,-0.006627,...,0.000000,0.000000,0.000000,-0.000537,0.012716,-0.006627,0.006627,0.006090,0.000537,True
1,1,established,False,250000,0.0,T4c,target,-1,721,-0.006627,...,0.012812,-0.009518,-0.009518,-0.018941,0.013349,-0.016145,0.016145,-0.002796,0.013349,True
2,1,established,False,250000,0.0,T4c,matched_random,0,721,-0.006627,...,0.001165,0.008796,0.004457,0.001702,-0.002638,0.002170,0.002170,-0.000468,0.001702,True
3,1,established,False,250000,0.0,T4c,matched_random,1,721,-0.006627,...,0.002253,0.001099,0.001099,-0.002790,0.008266,-0.005528,0.005528,0.002738,0.002790,True
4,1,established,False,250000,0.0,T4c,matched_random,2,721,-0.006627,...,-0.000726,0.007112,0.006142,0.001159,0.000189,0.000485,0.000485,0.000674,-0.000189,False
5,1,established,False,250000,0.0,T4c,matched_random,3,721,-0.006627,...,0.000967,0.015169,-0.001915,0.015580,-0.001504,0.008542,0.008542,0.007038,0.001504,True
6,1,established,False,250000,0.0,T4c,matched_random,4,721,-0.006627,...,-0.001769,0.007082,0.006172,0.002142,0.001232,0.000455,0.000455,0.001687,-0.001232,False
7,1,established,False,250000,0.0,T4c,matched_random,5,721,-0.006627,...,0.000115,0.004305,0.004305,-0.003992,0.000652,-0.002322,0.002322,-0.001670,0.000652,True
8,1,established,False,250000,0.0,T4c,matched_random,6,721,-0.006627,...,-0.007592,0.011226,0.002028,0.016252,0.007055,0.004599,0.004599,0.011653,-0.007055,False
9,1,established,False,250000,0.0,T4c,matched_random,7,721,-0.006627,...,-0.013356,0.006893,0.006360,0.013352,0.012819,0.000266,0.000266,0.013086,-0.012819,False


## 7. Statistics
Model/seed is the unit. Stored intervals retain the original inference choices. Fig3/S7 additionally replay the exact original bootstrap seed and RNG consumption order; cached null draws are not regenerated or tuned.


In [11]:
statistics = statistical_tables(tables)
for name, frame in statistics.items():
    print(name)
    display(frame)


data/precomputed/fig06/exp10_model_level_statistics_corrected.csv


,intervention_id,mean_loss,ci95_low,ci95_high,median_loss,sd_loss,n_loss_positive,p_signflip,mean_shape_residual,mean_propeller_loss,mean_specificity,kind,method,cell_types,family,inference_family,q_prespecified_family
0,control__all34__baseline,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,10,0.001951,0.000000e+00,1.000000,0.000000e+00,control,baseline,T1|T2|T2a|T3|T4a|T4b|T4c|T4d|T5a|T5b|T5c|T5d|T...,controls,controls,0.001951
1,control__all34__zero,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,10,0.001951,0.000000e+00,1.000000,0.000000e+00,control,zero,T1|T2|T2a|T3|T4a|T4b|T4c|T4d|T5a|T5b|T5c|T5d|T...,controls,controls,0.001951
2,control__noop,1.365574e-15,9.214851e-16,1.876277e-15,1.276756e-15,8.242778e-16,10,0.001951,8.918803e-16,0.000000,1.365574e-15,control,noop,NaN,controls,controls,0.001951
3,control__sham,1.365574e-15,9.214851e-16,1.876277e-15,1.276756e-15,8.242778e-16,10,0.001951,8.918803e-16,0.000000,1.365574e-15,control,sham,NaN,controls,controls,0.001951
4,group__T4_T5__baseline,6.281187e-01,2.636447e-01,9.094720e-01,8.146622e-01,5.613993e-01,9,0.006829,3.482151e-01,0.608594,1.952503e-02,group,baseline,T4a|T4b|T4c|T4d|T5a|T5b|T5c|T5d,groups,prespecified_groups,0.017073
5,group__T4_family__baseline,3.867609e-01,8.699287e-02,6.602980e-01,4.854555e-01,4.897838e-01,8,0.018537,4.633981e-01,0.523298,-1.365375e-01,group,baseline,T4a|T4b|T4c|T4d,groups,prespecified_groups,0.030894
6,group__T5_family__baseline,3.903721e-02,-7.895861e-02,1.805629e-01,1.367888e-02,2.212674e-01,5,0.323902,2.460667e-01,0.117896,-7.885919e-02,group,baseline,T5a|T5b|T5c|T5d,groups,prespecified_groups,0.323902
7,group__formal_candidates__baseline,1.310806e-01,-1.185251e-01,3.689800e-01,2.067920e-01,4.142433e-01,7,0.170732,3.661687e-01,0.296926,-1.658458e-01,group,baseline,T4a|TmY14|T2a|Tm16,groups,prespecified_groups,0.213415
8,group__formal_candidates_plus_T4b__baseline,4.943027e-01,2.894050e-01,7.101164e-01,4.870294e-01,3.597412e-01,10,0.001951,4.499603e-01,0.375332,1.189704e-01,group,baseline,T4a|TmY14|T2a|Tm16|T4b,groups,prespecified_groups,0.009756
9,single__T1__baseline,-2.943943e-01,-7.379929e-01,2.432923e-02,-2.709533e-02,6.672882e-01,3,0.887805,3.719104e-01,0.067833,-3.622274e-01,single,baseline,T1,screen,exploratory_34_screen,0.986341


data/precomputed/fig06/replacement_bootstrap.csv


,intervention,metric,n_models,mean,median,ci_low,ci_high
0,T4,MIDD_loss,10,-0.819937,0.864619,-3.453122,1.103754
1,T4,propeller_loss,10,0.596824,0.599281,0.520820,0.661208
2,T4,edge_loss,10,0.458687,0.566652,0.100500,0.752396
3,T4,propeller_magnitude_loss,10,0.286890,0.261829,0.216857,0.360765
4,T4,D_primary,10,-1.416760,0.203123,-3.987315,0.479301
5,T4,D_edge,10,-1.278624,0.291629,-3.951124,0.735470
6,T4,D_projected,10,0.255264,0.179512,-0.019905,0.638554
7,T4,MIDD_absolute_loss,10,-1.042770,0.742496,-3.582718,0.790762
8,T4_T5,MIDD_loss,10,-0.147103,1.014150,-2.753953,1.327386
9,T4_T5,propeller_loss,10,0.709927,0.696193,0.649792,0.776163


data/precomputed/fig06/denominator_audit.csv


,intervention,condition,q0,q5,q25,q50,q75,q95,q100,n,excluded
0,T4,MIDD,0.115196,0.116270,0.121877,0.157965,0.200315,0.235772,0.253071,10,0
1,T4,edge,0.759230,0.813264,0.953862,1.134334,1.260944,1.654653,1.857701,10,0
2,T4,propeller,9.085153,9.288611,10.627711,11.237086,11.801027,13.651906,13.796480,10,0
3,T4_T5,MIDD,0.115196,0.116270,0.121877,0.157965,0.200315,0.235772,0.253071,10,0
4,T4_T5,edge,0.759230,0.813264,0.953862,1.134334,1.260944,1.654653,1.857701,10,0
5,T4_T5,propeller,9.085153,9.288611,10.627711,11.237086,11.801027,13.651906,13.796480,10,0
6,T4a,MIDD,0.115196,0.116270,0.121877,0.157965,0.200315,0.235772,0.253071,10,0
7,T4a,edge,0.759230,0.813264,0.953862,1.134334,1.260944,1.654653,1.857701,10,0
8,T4a,propeller,9.085153,9.288611,10.627711,11.237086,11.801027,13.651906,13.796480,10,0
9,T4b,MIDD,0.115196,0.116270,0.121877,0.157965,0.200315,0.235772,0.253071,10,0


## 8. Figure generation
The shared plotting source preserves the adopted layout and labels; no manual PDF editing.


In [12]:
fig = render(FIGURE)
display(fig)


Saved S9 inputs 7 checks 2


<Figure size 1378x1560 with 8 Axes>

## 9. Numerical summary
Summaries remain labeled by their source; repeated conditions are not independent observations.


In [13]:
print("Figure:", FIGURE, "primary tables:", len(primary), "adopted tables:", len(adopted))


Figure: S9 primary tables: 1 adopted tables: 7


## 10. Reproducibility metadata and hashes
Record inputs, shared source versions, numerical summaries, software, outputs, and limitations.


In [14]:
record = write_reproduction_record(FIGURE, ROOT, manifest, integration_checks, primary, adopted)
print("Canonical notebook:", record["canonical_notebook"])
print("Model inference:", record["inference_performed"])


Canonical notebook: notebooks/supplementary/fig_S9.ipynb
Model inference: False
